In [5]:
from cosmo.models.tcn_new import DilatedCausalConvNet
import torch
model = DilatedCausalConvNet(
    config={},
    in_channels=8,
    residual_channels=64,
    skip_channels=64,
    out_channels=4,
    kernel_size=2,
    num_blocks=2,
    num_layers=4
)

In [6]:
model

DilatedCausalConvNet(
  (input_conv): CausalConv1d(
    (conv): ParametrizedConv1d(
      8, 64, kernel_size=(1,), stride=(1,)
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _WeightNorm()
        )
      )
    )
    (ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (relu): ReLU()
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (blocks): ModuleList(
    (0): ResidualBlock(
      (dilated_conv): CausalConv1d(
        (conv): ParametrizedConv1d(
          64, 128, kernel_size=(2,), stride=(1,), padding=(1,)
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
        (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (relu): ReLU()
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (residual_out): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
      (skip_out): Conv1d(64, 64, kernel_size=(1,), stride=(

In [29]:
print('Number of parameters:', sum(p.numel() for p in model.parameters()))

input_tensor = torch.randn(2, 9, 8)  # Batch size 512, 8 input channel, sequence length 64
output = model(input_tensor)
output.shape

Number of parameters: 206917


torch.Size([2, 4])

In [20]:
import torch.nn.functional as F
x = input_tensor
x = x.permute(0, 2, 1)
x = model.input_conv(x)

skip_connections = []
for block in model.blocks:
    x, skip = block(x)
    skip_connections.append(skip)

# skip_sum = torch.sum(torch.stack(skip_connections), dim=0)
# combined = model.alpha * skip_sum + (1 - model.alpha) * x
# x = F.relu(combined)
# x = F.relu(model.output_conv1(x))
# x = model.output_conv2(x)

# # x = torch.mean(x, dim=-1)  # Global average pooling along time dimension

In [21]:
x.shape

torch.Size([2, 64, 512])

In [22]:
skip_sum = torch.sum(torch.stack(skip_connections), dim=0)
skip_sum.shape

torch.Size([2, 64, 512])

In [23]:
combined = model.alpha * skip_sum + (1 - model.alpha) * x
combined.shape

torch.Size([2, 64, 512])

In [24]:
model.output_conv1(x).shape

torch.Size([2, 64, 512])

In [25]:
x = model.output_conv2(x)
x.shape

torch.Size([2, 4, 512])

In [26]:
x = torch.mean(x, dim=-1)
x.shape

torch.Size([2, 4])